# Falcon AI Trace Debugging

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/falcon-ai/context-aware-debugging)

Send two versions of the same research request. Open the first trace in Falcon AI, review the prompt change, then compare the second trace.

## Install

Use a fresh Python 3.12 kernel. The key cell uses existing environment variables or prompts without displaying your input.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "fi-instrumentation-otel", "traceai-openai", "openai"],
    check=True,
);

In [ ]:
import os
from getpass import getpass

for name in ['FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ").strip()
        if not value:
            raise ValueError(f"{name} is required")
        os.environ[name] = value
        del value

# For a self-hosted deployment, set FI_TRACES_URL to your HTTP collector.
os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com");

## Add tracing to your agent

The model receives no excerpt from the fictional report. Check whether it asks for the source or invents details.

In [ ]:
from fi_instrumentation import register, FITracer
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    verbose=False,
    project_type=ProjectType.OBSERVE,
    project_name="research-assistant-demo",
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
tracer = FITracer(trace_provider.get_tracer("research-assistant-demo"))

In [ ]:
from pathlib import Path
from uuid import UUID
from openai import OpenAI
from opentelemetry.trace import get_current_span

client = OpenAI()
question = "What does Ridgeline Outfitters’ internal Trail Materials Report say about recycled nylon?"
system_prompt = "You are a research assistant. Provide citations to support your claims."
trace_ids = []


# Replace this with your own agent's entry point.
# The @tracer.agent decorator makes each call show up as one parent span
# in your Future AGI Tracing project, with the OpenAI calls nested underneath.
@tracer.agent(name="my_agent")
def my_agent(user_message: str) -> str:
    trace_ids.append(str(UUID(int=get_current_span().get_span_context().trace_id)))
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content or ""


# This fictional report is not in the messages, and there is no retrieval tool.
print(my_agent(question))
trace_provider.force_flush()
print("Original trace:", trace_ids[-1])

## Ask Falcon AI what went wrong

Open the original trace under **Tracing → research-assistant-demo**, then open Falcon AI and ask: “What went wrong with this trace?” Check that the trace is attached.

## Drill into the failure mode

Run `/analyze-trace-errors` in the same conversation. Check any finding against the messages in the LLM span; there might be no error.

## Generate the prompt fix

Run `/fix-with-falcon`. Compare the suggested edit with the system message on the trace before applying it.

## Verify the fix

Try the example edit below, then compare the two traces. Did the second answer ask for the missing excerpt?

In [ ]:
system_prompt = ("Answer questions about internal reports only from excerpts supplied "
                 "in the conversation. If no excerpt is supplied, ask for it. Do not invent citations.")
print(my_agent(question))
trace_provider.force_flush()
print("Revised trace:", trace_ids[-1])
Path("research-traces.txt").write_text("\n".join(trace_ids))

`research-traces.txt` holds the original and revised trace IDs. Review both answers before deciding whether to keep the new prompt.